# Proportion Tests (Seed-Level): EEGFormer vs Other Models

This notebook performs Bernoulli proportion tests using exactly one seed (the best EEGFormer seed), so each model is evaluated over 120 subject-level outcomes.

- Success = 1 if a subject is well classified
- Failure = 0 otherwise

Data source: `results/hypothesis tests/*_10seeds/all_fold_results.csv` and partition files.

Statistical test for each comparison:
- `statsmodels.stats.proportion.proportions_ztest(count, nobs, alternative='two-sided', prop_var=False)`

Goal: compare EEGFormer against all other models using one-seed totals (120 subjects) to avoid cross-seed dependence.

In [7]:
from pathlib import Path

import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportion_confint, proportions_ztest

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

In [8]:
BASE_DIR = Path('../results/hypothesis tests')
OUT_DIR = BASE_DIR / 'proportion_test_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = [
    {'name': 'ShallowConvNet 2017', 'folder': 'shallowconvnet_10seeds'},
    {'name': 'EEGNet 2017', 'folder': 'eegnet_10seeds'},
    {'name': 'CNN--LSTM 2022', 'folder': 'cnn_lstm_eegnet_10seeds'},
    {'name': 'Multi-Stream 2023', 'folder': 'multistream_10seeds'},
    {'name': 'IM--CBGT 2024', 'folder': 'imcbgt_10seeds'},
    {'name': 'T-GARNet 2025', 'folder': 'tgarnet_10seeds'},
    {'name': 'EEG-Former 2026 (This work)', 'folder': 'eegformer_10seeds'},
]

TARGET_MODEL = 'EEG-Former 2026 (This work)'
USER_GUESS_SEED = 123
EXPECTED_SUBJECTS = 120

In [9]:
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [str(c).strip().lower().replace(' ', '_').replace('-', '_') for c in out.columns]
    return out


def holm_bonferroni(p_values: pd.Series) -> pd.Series:
    m = len(p_values)
    ordered = np.argsort(p_values.values)
    adjusted = np.empty(m, dtype=float)
    running_max = 0.0
    for rank, idx in enumerate(ordered, start=1):
        raw = float(p_values.iloc[idx])
        value = (m - rank + 1) * raw
        running_max = max(running_max, value)
        adjusted[idx] = min(1.0, running_max)
    return pd.Series(adjusted, index=p_values.index)


def find_best_seed_for_model(base_dir: Path, folder: str) -> int:
    seed_path = base_dir / folder / 'all_seed_summaries.csv'
    if not seed_path.exists():
        raise FileNotFoundError(f'Missing file: {seed_path}')
    seed_df = clean_columns(pd.read_csv(seed_path))
    needed = {'seed', 'mean_accuracy'}
    missing = needed - set(seed_df.columns)
    if missing:
        raise ValueError(f'Missing columns {sorted(missing)} in {seed_path.name}')
    seed_df['seed'] = pd.to_numeric(seed_df['seed'], errors='raise').astype(int)
    seed_df['mean_accuracy'] = pd.to_numeric(seed_df['mean_accuracy'], errors='raise').astype(float)
    return int(seed_df.sort_values(['mean_accuracy', 'seed'], ascending=[False, True]).iloc[0]['seed'])

In [10]:
target_cfg = next(cfg for cfg in MODELS if cfg['name'] == TARGET_MODEL)
best_seed = find_best_seed_for_model(BASE_DIR, target_cfg['folder'])

print(f'User guess seed: {USER_GUESS_SEED}')
print(f'Best seed for {TARGET_MODEL}: {best_seed}')

seed_to_use = best_seed
print(f'Using seed: {seed_to_use}')

model_rows = []
availability_rows = []

for cfg in MODELS:
    model_name = cfg['name']
    folder = cfg['folder']

    fold_path = BASE_DIR / folder / 'all_fold_results.csv'
    part_path = BASE_DIR / folder / 'partitions' / 'fixed_partitions_summary.csv'

    if not fold_path.exists():
        raise FileNotFoundError(f'Missing file: {fold_path}')
    if not part_path.exists():
        raise FileNotFoundError(f'Missing file: {part_path}')

    df = clean_columns(pd.read_csv(fold_path))
    parts = clean_columns(pd.read_csv(part_path))

    needed = {'seed', 'fold', 'accuracy'}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f'{model_name}: missing columns {sorted(missing)} in {fold_path.name}')
    if 'n_test_subjects' not in parts.columns:
        raise ValueError(f'{model_name}: missing n_test_subjects in {part_path.name}')

    n_test_subjects = int(round(float(parts['n_test_subjects'].median())))

    tmp = df[['seed', 'fold', 'accuracy']].copy()
    tmp['seed'] = pd.to_numeric(tmp['seed'], errors='raise').astype(int)
    tmp['fold'] = pd.to_numeric(tmp['fold'], errors='raise').astype(int)
    tmp['accuracy'] = pd.to_numeric(tmp['accuracy'], errors='raise').astype(float)
    tmp = tmp[tmp['seed'] == seed_to_use].copy()

    if tmp.empty:
        raise ValueError(f'{model_name}: seed {seed_to_use} not found in all_fold_results.csv')

    tmp['n_test_subjects'] = n_test_subjects
    tmp['n_correct_subjects'] = np.rint(tmp['accuracy'] * n_test_subjects).astype(int)
    tmp['model'] = model_name

    availability_rows.append({
        'model': model_name,
        'folder': folder,
        'seed_used': seed_to_use,
        'n_folds_found': int(tmp['fold'].nunique()),
        'n_rows_found': int(len(tmp)),
        'n_test_subjects_per_fold': n_test_subjects,
        'n_subjects_total': int(n_test_subjects * tmp['fold'].nunique()),
    })

    model_rows.append(tmp)

raw = pd.concat(model_rows, ignore_index=True)
availability = pd.DataFrame(availability_rows).sort_values('model').reset_index(drop=True)
display(availability)

if not (availability['n_subjects_total'] == EXPECTED_SUBJECTS).all():
    print('Warning: At least one model does not sum to 120 subjects for this seed.')

User guess seed: 123
Best seed for EEG-Former 2026 (This work): 2718
Using seed: 2718


,model,folder,seed_used,n_folds_found,n_rows_found,n_test_subjects_per_fold,n_subjects_total
0,CNN--LSTM 2022,cnn_lstm_eegnet_10seeds,2718,5,5,24,120
1,EEG-Former 2026 (This work),eegformer_10seeds,2718,5,5,24,120
2,EEGNet 2017,eegnet_10seeds,2718,5,5,24,120
3,IM--CBGT 2024,imcbgt_10seeds,2718,5,5,24,120
4,Multi-Stream 2023,multistream_10seeds,2718,5,5,24,120
5,ShallowConvNet 2017,shallowconvnet_10seeds,2718,5,5,24,120
6,T-GARNet 2025,tgarnet_10seeds,2718,5,5,24,120


In [11]:
summary = (
    raw.groupby('model', as_index=False)
    .agg(
        successes=('n_correct_subjects', 'sum'),
        total=('n_test_subjects', 'sum')
    )
)
summary['proportion'] = summary['successes'] / summary['total']

cis = summary.apply(
    lambda r: proportion_confint(
        count=int(r['successes']),
        nobs=int(r['total']),
        alpha=0.05,
        method='wilson'
    ),
    axis=1
 )
summary['ci_low'] = [float(ci[0]) for ci in cis]
summary['ci_high'] = [float(ci[1]) for ci in cis]

display(summary.sort_values('proportion', ascending=False).reset_index(drop=True))

,model,successes,total,proportion,ci_low,ci_high
0,EEG-Former 2026 (This work),104,120,0.866667,0.794352,0.916234
1,CNN--LSTM 2022,101,120,0.841667,0.765907,0.896230
2,EEGNet 2017,96,120,0.800000,0.719633,0.861755
3,ShallowConvNet 2017,96,120,0.800000,0.719633,0.861755
4,IM--CBGT 2024,87,120,0.725000,0.639070,0.796971
5,T-GARNet 2025,85,120,0.708333,0.621558,0.782184
6,Multi-Stream 2023,82,120,0.683333,0.595521,0.759772


In [12]:
target_row = summary.loc[summary['model'] == TARGET_MODEL]
if target_row.empty:
    raise ValueError(f'Target model not found: {TARGET_MODEL}')

x_t = int(target_row['successes'].iloc[0])
n_t = int(target_row['total'].iloc[0])
p_t = x_t / n_t

comparisons = []
for _, row in summary.iterrows():
    model = row['model']
    if model == TARGET_MODEL:
        continue

    x_o = int(row['successes'])
    n_o = int(row['total'])
    p_o = x_o / n_o

    z_stat, p_val = proportions_ztest(
        count=[x_t, x_o],
        nobs=[n_t, n_o],
        value=0,
        alternative='two-sided',
        prop_var=False
    )

    comparisons.append({
        'seed_used': seed_to_use,
        'target_model': TARGET_MODEL,
        'other_model': model,
        'target_successes': x_t,
        'target_total': n_t,
        'target_proportion': p_t,
        'other_successes': x_o,
        'other_total': n_o,
        'other_proportion': p_o,
        'difference_target_minus_other': p_t - p_o,
        'z_stat': float(z_stat),
        'p_value_two_sided': float(p_val),
    })

comp_df = pd.DataFrame(comparisons).sort_values('p_value_two_sided').reset_index(drop=True)
comp_df['p_value_holm'] = holm_bonferroni(comp_df['p_value_two_sided'])
comp_df['significant_0_05_holm'] = comp_df['p_value_holm'] < 0.05

display(comp_df)

summary_out = OUT_DIR / f'model_subject_proportions_seed_{seed_to_use}.csv'
comp_out = OUT_DIR / f'eegformer_vs_others_two_proportion_seed_{seed_to_use}.csv'
avail_out = OUT_DIR / f'availability_check_seed_{seed_to_use}.csv'

summary.to_csv(summary_out, index=False)
comp_df.to_csv(comp_out, index=False)
availability.to_csv(avail_out, index=False)

print('Saved:')
print(summary_out)
print(comp_out)
print(avail_out)

,seed_used,target_model,other_model,target_successes,target_total,target_proportion,other_successes,other_total,other_proportion,difference_target_minus_other,z_stat,p_value_two_sided,p_value_holm,significant_0_05_holm
0,2718,EEG-Former 2026 (This work),Multi-Stream 2023,104,120,0.866667,82,120,0.683333,0.183333,3.400752,0.000672,0.004032,True
1,2718,EEG-Former 2026 (This work),T-GARNet 2025,104,120,0.866667,85,120,0.708333,0.158333,2.998080,0.002717,0.013584,True
2,2718,EEG-Former 2026 (This work),IM--CBGT 2024,104,120,0.866667,87,120,0.725000,0.141667,2.722324,0.006482,0.025930,True
3,2718,EEG-Former 2026 (This work),EEGNet 2017,104,120,0.866667,96,120,0.800000,0.066667,1.385641,0.165857,0.497570,False
4,2718,EEG-Former 2026 (This work),ShallowConvNet 2017,104,120,0.866667,96,120,0.800000,0.066667,1.385641,0.165857,0.497570,False
5,2718,EEG-Former 2026 (This work),CNN--LSTM 2022,104,120,0.866667,101,120,0.841667,0.025000,0.548676,0.583228,0.583228,False


Saved:
..\results\hypothesis tests\proportion_test_outputs\model_subject_proportions_seed_2718.csv
..\results\hypothesis tests\proportion_test_outputs\eegformer_vs_others_two_proportion_seed_2718.csv
..\results\hypothesis tests\proportion_test_outputs\availability_check_seed_2718.csv


## Is This Better?

Yes, this is a better idea than pooling 10 seeds into 1200 outcomes.

Why it is better:
- You no longer duplicate the same subjects across seeds.
- Using one seed gives one Bernoulli outcome per subject, so the within-model sample is cleaner (120 subjects).

Still important:
- Across models, the same 120 subjects are evaluated, so results are naturally paired by subject.
- A two-sample z-test treats groups as independent, so it is still approximate for model-vs-model inference.

Best strict inference (if per-subject predictions are available):
- Use McNemar test between EEGFormer and each model on paired subject correctness.
- Optionally report this z-test as a secondary descriptive analysis.

Practical conclusion:
- This seed-level version is statistically more defensible than the previous 1200-outcome pooled analysis.
- For final claims, paired subject-level tests are still preferable.